Import Libraries

In [1]:
import jax
import flax
import optax
from jax import lax, random, numpy as jnp
from jax import random, grad, vmap, hessian, jacfwd, jit
from jax import config
from flax import linen as nn
from evojax.util import get_params_format_fn

import time
import numpy as np
import pandas as pd
from scipy import io
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.optimize import minimize_scalar



from matplotlib import rcParams
import math
# choose GPU
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
#jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

# config = {
#     "font.family": 'Times New Roman',
#     "font.size": 14,
#     "mathtext.fontset": 'stix',
#     "font.serif": ['SimSun'],
# }
# rcParams.update(config)

# rcParams['axes.unicode_minus'] = False

Problem: Navier-Stokes Equation

        u*u_x + v*u_y - 1/Re*(u_xx+u_yy) + p_x = 0
        v*v_x + v*v_y - 1/Re*(v_xx+v_yy) + p_y = 0
        u_x + v_y = 0

In [2]:
# parameter
Re = 500

In [3]:
sim = pd.read_csv('naca0012_RE500_801x401.csv',sep=',')
print("x: ", sim['x'].shape, ", y: " , sim['y'].shape,", u: ", sim['u'].shape, ", v: " , sim['v'].shape, "p: ", sim['p'].shape)

x:  (321201,) , y:  (321201,) , u:  (321201,) , v:  (321201,) p:  (321201,)


In [4]:
sim

,x,y,u,v,p,voricity,phi,Uc,Vc,Unnamed: 9
0,-3.00,-2.0,1.00000,0.0,0.021398,-0.000004,-3.60281,0.00000,0.0,NaN
1,-2.99,-2.0,1.00000,0.0,0.021398,-0.000004,-3.59448,1.00000,0.0,NaN
2,-2.98,-2.0,1.00000,0.0,0.021398,-0.000004,-3.58615,1.00000,0.0,NaN
3,-2.97,-2.0,1.00000,0.0,0.021397,-0.000004,-3.57784,1.00000,0.0,NaN
4,-2.96,-2.0,1.00000,0.0,0.021396,-0.000004,-3.56953,1.00000,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...
321196,4.96,2.0,1.02711,0.0,-0.006125,0.000004,-4.43672,1.02711,0.0,NaN
321197,4.97,2.0,1.02710,0.0,-0.006125,0.000004,-4.44565,1.02710,0.0,NaN
321198,4.98,2.0,1.02710,0.0,-0.006124,0.000004,-4.45458,1.02710,0.0,NaN
321199,4.99,2.0,1.02710,0.0,-0.006124,0.000004,-4.46352,1.02710,0.0,NaN


In [5]:
sim_x = sim['x'].values.reshape(-1,1)
sim_y = sim['y'].values.reshape(-1,1)
sim_u = sim['u'].values.reshape(-1,1)
sim_v = sim['v'].values.reshape(-1,1)
sim_p = sim['p'].values.reshape(-1,1)


In [6]:
x_l, x_u, y_l, y_u = np.min(sim_x), np.max(sim_x), np.min(sim_y), np.max(sim_y)

ext = [x_l, x_u, y_l, y_u]
print (x_l, x_u, y_l, y_u)


-3.0 5.0 -2.0 2.0


In [7]:
data_X, data_Y = np.hstack([sim_x, sim_y]), np.hstack([sim_u, sim_v, sim_p])
data_X

array([[-3.  , -2.  ],
       [-2.99, -2.  ],
       [-2.98, -2.  ],
       ...,
       [ 4.98,  2.  ],
       [ 4.99,  2.  ],
       [ 5.  ,  2.  ]])

In [8]:
data_Y

array([[ 1.        ,  0.        ,  0.0213979 ],
       [ 1.        ,  0.        ,  0.0213978 ],
       [ 1.        ,  0.        ,  0.0213976 ],
       ...,
       [ 1.0271    ,  0.        , -0.00612435],
       [ 1.0271    ,  0.        , -0.00612412],
       [ 1.        ,  0.        , -0.00612408]])

In [9]:
x_unique, y_unique = np.unique(sim_x), np.unique(sim_y)
dx =  x_unique[2]-x_unique[1]
dy =  y_unique[2]-y_unique[1]
print ('dx:',dx, 'dy:',dy)
print ( 'x_unique shape: ', x_unique.shape)
print ( 'y_unique shape: ', y_unique.shape)

dx: 0.010000000000000231 dy: 0.010000000000000009
x_unique shape:  (801,)
y_unique shape:  (401,)


In [10]:
data_Y_mat = data_Y[:,0].reshape(y_unique.size, x_unique.size)
data_Y_mat.shape

(401, 801)

In [11]:
def naca0012_thickness(x, thickness=0.12):
        return 5 * thickness * (0.2969 * np.sqrt(x) - 0.1260 * x - 
                              0.3516 * x**2 + 0.2843 * x**3 - 0.1015 * x**4)
        

def sample_points_on_airfoil(num_points=100, thickness=0.12, equal_spacing=True):

    
    x_dense = np.linspace(0, 1, 1000)
    y_dense = naca0012_thickness(x_dense, thickness)
    
    x_upper = x_dense[::-1]
    y_upper = y_dense[::-1]
    
    x_lower = x_dense[1:]
    y_lower = -y_dense[1:]
    
    x_all = np.concatenate([x_upper, x_lower])
    y_all = np.concatenate([y_upper, y_lower])
    
    if equal_spacing:
        ds = np.sqrt(np.diff(x_all)**2 + np.diff(y_all)**2) # Calculate the distance between consecutive points (dx**2 + dy**2)**0.5
        s = np.concatenate([[0], np.cumsum(ds)])  # Calculate the cumulative sum of distances from 0

        total_length = s[-1]  # Total length of the airfoil contour
        
        # 
        s_uniform = np.linspace(0, total_length, num_points)

        fx = interp1d(s, x_all)  # the element of s is the cumulative distance
        fy = interp1d(s, y_all)
        x_sampled = fx(s_uniform)
        y_sampled = fy(s_uniform)
    else:
        half_points = num_points // 2
        
        x_upper_sampled = np.linspace(0, 1, half_points)
        y_upper_sampled = naca0012_thickness(x_upper_sampled, thickness)
        
        x_lower_sampled = x_upper_sampled
        y_lower_sampled = -y_upper_sampled
        
        x_sampled = np.concatenate([x_upper_sampled, x_lower_sampled])
        y_sampled = np.concatenate([y_upper_sampled, y_lower_sampled])
    
    return x_sampled, y_sampled

def filter_points_inside_airfoil(data):
    x = data[:, 0]  
    y = data[:, 1]  
    
    inside_airfoil = np.zeros(len(x), dtype=bool)
    
    x_valid = (x >= 0) & (x <= 1)
    
    valid_indices = np.where(x_valid)[0]
    for i in valid_indices:
        thickness = naca0012_thickness(x[i], 0.12)
        if -thickness <= y[i] <= thickness:
            inside_airfoil[i] = True
    
    return inside_airfoil

def get_valid_points_mask(data, dx=dx, dy=dy, thickness=0.12):
  
    def naca0012_thickness_at_x(x, thickness=0.12):
        mask = (x >= 0) & (x <= 1)
        y_t = np.zeros_like(x)
        y_t[mask] = 5 * thickness * (0.2969 * np.sqrt(x[mask]) - 0.1260 * x[mask] - 
                                    0.3516 * x[mask]**2 + 0.2843 * x[mask]**3 - 
                                    0.1015 * x[mask]**4)
        return y_t  
    
    def is_outside_airfoil(x_pt, y_pt):
        if x_pt < 0 or x_pt > 1:
            return True
        
        half_thickness = naca0012_thickness_at_x(np.array([x_pt]))[0]
        
        return y_pt > half_thickness or y_pt < -half_thickness
    
    valid_mask = np.zeros(len(data), dtype=bool)
    
    for i in range(len(data)):
        x, y = data[i]
        
        surrounding_points = [
            [x, y],           
            [x + dx, y],      
            [x - dx, y],     
            [x, y + dy],      #
            [x, y - dy]       #
        ]
        
        all_outside = True
        for point in surrounding_points:
            x_pt, y_pt = point
            if not is_outside_airfoil(x_pt, y_pt):
                all_outside = False
                break
        
        valid_mask[i] = all_outside
    
    return valid_mask


In [12]:

def naca0012_derivative(x, thickness=0.12):

    return 5 * thickness * (0.2969 * 0.5 / np.sqrt(x) - 0.1260 -
                          0.3516 * 2 * x + 0.2843 * 3 * x**2 - 0.1015 * 4 * x**3)
    

def find_closest_point(x0, y0):
    
    if abs(y0) < 1e-9:
        # raise ValueError("The external point must have y = 0.")
    
        if x0  < 0.0:
            return abs(x0 - 0), (0, 0), True  

        if x0 > 1.0:
            return abs(x0 - 1), (1, 0), True 
    
    
    def distance_to_upper(x):
        y = naca0012_thickness(x)
        return (x - x0)**2 + (y - y0)**2
    
    def distance_to_lower(x):
        y = -naca0012_thickness(x)
        return (x - x0)**2 + (y - y0)**2
    
    bounds = [(0, 1)]
    
    result_upper = minimize(distance_to_upper, 0.5, bounds=bounds)
    if result_upper.success:
        x_upper = result_upper.x[0]
        y_upper = naca0012_thickness(x_upper)
        dist_upper = np.sqrt(result_upper.fun)
    else:
        x_upper, y_upper, dist_upper = np.nan, np.nan, float("inf")
    result_lower = minimize(distance_to_lower, 0.5, bounds=bounds)

    if result_lower.success:
        x_lower = result_lower.x[0]
        y_lower = -naca0012_thickness(x_lower)
        dist_lower = np.sqrt(result_lower.fun)
        
    else:
        x_lower, y_lower, dist_lower = np.nan, np.nan, float("inf")
    
    if dist_upper <= dist_lower:
        return dist_upper, (x_upper, y_upper), True  
    else:
        return dist_lower, (x_lower, y_lower), False  



def calculate_normal_vector(x_int, is_upper):
    if x_int < 1e-10:  
        tangent_slope = np.inf if is_upper else -np.inf
        normal_x = -1 if is_upper else 1
        normal_y = 0
    else:
        tangent_slope = naca0012_derivative(x_int)
        if not is_upper:
            tangent_slope = -tangent_slope
            
        if abs(tangent_slope) < 1e-10: 
            normal_x = 0
            normal_y = 1 if is_upper else -1
        else:
            normal_slope = -1 / tangent_slope
            norm = np.sqrt(1 + normal_slope**2)
            normal_x = 1 / norm
            normal_y = normal_slope / norm
            
            if (is_upper and normal_y < 0) or (not is_upper and normal_y > 0):
                normal_x = -normal_x
                normal_y = -normal_y
    
    return normal_x, normal_y

In [13]:
naca0012_x_num = 100
naca0012_x = jnp.linspace(0, 1, naca0012_x_num).reshape(-1,1)
airfoil_x,airfoil_y = sample_points_on_airfoil(num_points=400, thickness=0.12, equal_spacing=True)
y_t = naca0012_thickness(naca0012_x, thickness=0.12)
x_upper, y_upper = naca0012_x, +y_t
x_lower, y_lower = naca0012_x, -y_t

air_bool = filter_points_inside_airfoil(data_X)
print("the points inside airfoil are: ", np.sum(air_bool))
print("the points outside airfoil are: ", np.sum(~air_bool))
print("the total number of points is: ", len(data_X))

data_X_airfoil = data_X[air_bool]
data_XX,data_YY = data_X[~air_bool], data_Y[~air_bool]
sample_points_outside_airfoil_bool = get_valid_points_mask(data_X, dx=dx, dy=dy, thickness=0.12)  
# data_X = data_X[sample_points_outside_airfoil_bool]
# data_Y = data_Y[sample_points_outside_airfoil_bool]

the points inside airfoil are:  807
the points outside airfoil are:  320394
the total number of points is:  321201


In [14]:
airfoil_x = airfoil_x.reshape(-1,1)
airfoil_y = airfoil_y.reshape(-1,1)


In [15]:

airfoil_shape = np.hstack([airfoil_x,airfoil_y])

airfoil_shape_value = np.zeros((len(airfoil_shape[:,0]),3))
print('the shape of airfoil shape is: ', airfoil_shape.shape)   
print("the points of fluid domain is: ", data_X.shape)

the shape of airfoil shape is:  (400, 2)
the points of fluid domain is:  (321201, 2)


In [16]:
print('data_X shape:',data_X.shape,'data_Y shape:',data_Y.shape)

l_bc = (data_X[:,0] == x_l)
r_bc = (data_X[:,0] == x_u)
tb_bc = (data_X[:,1] == y_l) | (data_X[:,1] == y_u)
data_X_l_BC,data_Y_l_BC = data_X[l_bc],data_Y[l_bc]
data_X_r_BC, data_Y_r_BC = data_X[r_bc], data_Y[r_bc]
data_X_tb_BC, data_Y_tb_BC = data_X[tb_bc], data_Y[tb_bc]
data_X_airfoil_BC,data_Y_airfoil_BC = airfoil_shape,airfoil_shape_value
data_X, data_Y = data_X[sample_points_outside_airfoil_bool],data_Y[sample_points_outside_airfoil_bool]
set_XX = set(map(tuple, data_XX))
set_X = set(map(tuple, data_X))

ad_X_points = np.array(list(set_XX - set_X))
point_to_y = {tuple(pt): y for pt, y in zip(data_XX, data_YY)}

ad_Y_points = np.array([point_to_y[tuple(pt)] for pt in ad_X_points])
print('data_X_l_BC shape:',data_X_l_BC.shape,'data_Y_l_BC shape:',data_Y_l_BC.shape)
print('data_X_r_BC shape:',data_X_r_BC.shape,'data_Y_r_BC shape:',data_Y_r_BC.shape)
print('data_X_tb_BC shape:',data_X_tb_BC.shape,'data_Y_tb_BC shape:',data_Y_tb_BC.shape)
print('data_X_naca_BC shape:',data_X_airfoil_BC.shape,'data_Y_naca_BC shape:',data_Y_airfoil_BC.shape)
print('the data shape after remove the airfoil:')
print('data_X shape:',data_X.shape,'data_Y shape:',data_Y.shape)
print("the points near the airfoil are: ", ad_X_points.shape)


data_X shape: (321201, 2) data_Y shape: (321201, 3)
data_X_l_BC shape: (401, 2) data_Y_l_BC shape: (401, 3)
data_X_r_BC shape: (401, 2) data_Y_r_BC shape: (401, 3)
data_X_tb_BC shape: (1602, 2) data_Y_tb_BC shape: (1602, 3)
data_X_naca_BC shape: (400, 2) data_Y_naca_BC shape: (400, 3)
the data shape after remove the airfoil:
data_X shape: (320190, 2) data_Y shape: (320190, 3)
the points near the airfoil are:  (204, 2)


In [17]:
point_ad = np.hstack([ad_X_points])   


In [18]:
# convert to jnp
data_X, data_Y= jnp.array(data_X), jnp.array(data_Y)
data_X_l_BC, data_Y_l_BC = jnp.array(data_X_l_BC), jnp.array(data_Y_l_BC)
data_X_r_BC, data_Y_r_BC = jnp.array(data_X_r_BC), jnp.array(data_Y_r_BC)
data_X_tb_BC, data_Y_tb_BC = jnp.array(data_X_tb_BC), jnp.array(data_Y_tb_BC)
data_X_airfoil_BC, data_Y_airfoil_BC = jnp.array(data_X_airfoil_BC), jnp.array(data_Y_airfoil_BC)
point_ad = jnp.array(point_ad) # 
print(data_X.shape, data_Y.shape)
print(data_X_l_BC.shape, data_Y_l_BC.shape)
print(data_X_r_BC.shape, data_Y_r_BC.shape)
print(data_X_tb_BC.shape, data_Y_tb_BC.shape)
print(data_X_airfoil_BC.shape, data_Y_airfoil_BC.shape)

(320190, 2) (320190, 3)
(401, 2) (401, 3)
(401, 2) (401, 3)
(1602, 2) (1602, 3)
(400, 2) (400, 3)


In [19]:
data_X_airfoil_BC[:,0].min()

Array(0.00045277, dtype=float32)

In [20]:
data_X_airfoil_BC[:,0].max()

Array(1., dtype=float32)

In [21]:
BS_ALL = int(len(data_X)*0.04)
BS_l_BC = int(len(data_X_l_BC)*0.1)
BS_r_BC = int(len(data_X_r_BC)*0.1)
BS_tb_BC = int(len(data_X_tb_BC)*0.1)
BS_airfoil_BC = int(len(data_X_airfoil_BC)*0.1)
BS_ad = int(len(point_ad)*0.2)



BS_PDE = BS_ALL - BS_l_BC - BS_r_BC - BS_tb_BC - BS_airfoil_BC - BS_ad # - BS_BC

n_all, n_l_BC, n_r_BC, n_tb_BC, n_airfoil_BC, n_ad  = len(data_X), len(data_X_l_BC), len(data_X_r_BC), len(data_X_tb_BC), len(data_X_airfoil_BC), len(point_ad)
print('The total number of samples:')
print('n_all: ',n_all, 'n_l_BC: ',n_l_BC, 'n_r_BC: ',n_r_BC, 'n_tb_BC: ',n_tb_BC, 'n_airfoil_BC: ',n_airfoil_BC, 'n_ad: ',n_ad)
print('The number of samples in each batch:')
print('BS_ALL: ',BS_ALL, 'BS_l_BC: ',BS_l_BC, 'BS_r_BC: ',BS_r_BC, 'BS_tb_BC: ',BS_tb_BC, 'BS_airfoil_BC: ',BS_airfoil_BC, 'BS_ad: ',BS_ad)

The total number of samples:
n_all:  320190 n_l_BC:  401 n_r_BC:  401 n_tb_BC:  1602 n_airfoil_BC:  400 n_ad:  204
The number of samples in each batch:
BS_ALL:  12807 BS_l_BC:  40 BS_r_BC:  40 BS_tb_BC:  160 BS_airfoil_BC:  40 BS_ad:  40


DNN / PINN   

In [22]:
nn_acf = nn.silu

class PINN(nn.Module):
    """PINNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]     


    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[:,0:1], inputs[:,1:2]
         
        def get_uvp(x, y):
            inputs = jnp.hstack([x,y ])

            # feature mapping
            hidden = self.feats(inputs)
            hidden = jnp.sin(jnp.pi*hidden)

            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p) 


            return (u, v, p)  
    
        u, v, p = get_uvp(x, y)

        xE, xW = x + dx, x - dx
        yN, yS = y + dy, y - dy
        xe, xw = x + 0.5*dx, x - 0.5*dx
        yn, ys = y + 0.5*dy, y - 0.5*dy
        # obtain u, v, p neighbour
        uE, vE, pE = get_uvp(xE, y)
        uW, vW, pW = get_uvp(xW, y)
        uN, vN, pN = get_uvp(x, yN)
        uS, vS, pS = get_uvp(x, yS)
        ue, ve, pe = get_uvp(xe, y)
        uw, vw, pw = get_uvp(xw, y)
        un, vn, pn = get_uvp(x, yn)
        us, vs, ps= get_uvp(x, ys)
        
        uWbc, vWbc = 1, 0
        vNbc = 0
        vSbc = 0
        pEbc = 0

         
        xB_E, xB_W = abs(xE-x_u)<0.01*dx, abs(xW-x_l)<0.01*dx
        yB_N, yB_S = abs(yN-y_u)<0.01*dy, abs(yS-y_l)<0.01*dy           
        # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
        pE = jnp.where(xB_E, pEbc, pE)
        uW, vW = jnp.where(xB_W, uWbc, uW), jnp.where(xB_W, vWbc, vW)
        vN = jnp.where(yB_N, vNbc, vN)
        vS = jnp.where(yB_S, vSbc, vS)
         
        xB_e, xB_w = abs(xe-x_u)<0.01*dx, abs(xw-x_l)<0.01*dx
        yB_n, yB_s = abs(yn-y_u)<0.01*dy, abs(ys-y_l)<0.01*dy           
        # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
        
        pe = jnp.where(xB_e, pEbc, pe)
        uw, vw = jnp.where(xB_w, uWbc, uw), jnp.where(xB_w, vWbc, vw)
        vn = jnp.where(yB_n, vNbc, vn)
        vs = jnp.where(yB_s, vSbc, vs)  
        
        ae = ue
        aw = -uw
        an = vn
        aas = -vs
        
        y_top_S = (y_u - dy)*jnp.ones_like(x)
        y_bottom_N = (y_l + dy)*jnp.ones_like(x)

        u_top_S,_,_ = get_uvp(x, y_top_S)
        u_bottom_N,_,_ =  get_uvp(x, y_bottom_N)

        top_bc_u = u_top_S - u
        bottom_bc_u = u_bottom_N - u

        
        source_x = (pe - pw)
        source_y = (pn - ps) 

        bc  = (x == x_l) | (x == x_u) | (y == y_l) | (y == y_u)
        l_bc = (x == x_l)
        t_bc = (y == y_u)
        b_bc = (y == y_l)  
        nbc = (~bc)
        uv_bc = (x == x_l) | (y == y_l) | (y == y_u)
        p_bc = (x == x_u)
        
        div = (uE - uW + vN - vS)/2

        mom_x = ae*ue + aw*uw + an*un + aas*us + source_x - (uE + uW + uN + uS - 4*u)/(dy*Re) 
        mom_y = ae*ve + aw*vw + an*vn + aas*vs + source_y - (vE + vW + vN + vS - 4*v)/(dx*Re) 

        res_u = ( source_x - (uE + uW + uN + uS)/(dx*Re))/dx
        res_v = ( source_y - (vE + vW + vN + vS)/(dx*Re))/dx
        
        residuals_continuity = div/dx
        residuals_momentum_1 = mom_x/dx
        residuals_momentum_2 = mom_y/dy 
        res_c = (ue - uw + vn - vs)/dx
        res_p = pE + pW + pN + pS
        
        outputs = jnp.hstack([u, v, p, residuals_continuity, residuals_momentum_1, residuals_momentum_2, uv_bc,p_bc, nbc,res_u,res_v,top_bc_u,bottom_bc_u,l_bc,t_bc,b_bc,res_c,res_p])

        return outputs    

  
    
class DNN(nn.Module):
    """DNNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]    


    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[:,0:1], inputs[:,1:2]

        def get_uvp(x, y):
            inputs = jnp.hstack([x,y ])

            # feature mapping
            hidden = self.feats(inputs)
            hidden = jnp.sin(jnp.pi*hidden)

            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p)

                
            return (u, v, p)  
            
        u, v, p = get_uvp(x, y)

        xE, xW = x + dx, x - dx
        yN, yS = y + dy, y - dy
        xe, xw = x + 0.5*dx, x - 0.5*dx
        yn, ys = y + 0.5*dy, y - 0.5*dy
        
        # obtain u, v, p neighbour
        uE, vE, pE = get_uvp(xE, y)
        uW, vW, pW = get_uvp(xW, y)
        uN, vN, pN = get_uvp(x, yN)
        uS, vS, pS = get_uvp(x, yS)
        ue, ve, pe = get_uvp(xe, y)
        uw, vw, pw = get_uvp(xw, y)
        un, vn, pn = get_uvp(x, yn)
        us, vs, ps= get_uvp(x, ys)
        
        
        uWbc, vWbc = 1, 0
        vNbc = 0
        vSbc = 0
        pEbc = 0

         
        xB_E, xB_W = abs(xE-x_u)<0.01*dx, abs(xW-x_l)<0.01*dx
        yB_N, yB_S = abs(yN-y_u)<0.01*dy, abs(yS-y_l)<0.01*dy           
        # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
        pE = jnp.where(xB_E, pEbc, pE)
        uW, vW = jnp.where(xB_W, uWbc, uW), jnp.where(xB_W, vWbc, vW)
        vN = jnp.where(yB_N, vNbc, vN)
        vS = jnp.where(yB_S, vSbc, vS)
         
        xB_e, xB_w = abs(xe-x_u)<0.01*dx, abs(xw-x_l)<0.01*dx
        yB_n, yB_s = abs(yn-y_u)<0.01*dy, abs(ys-y_l)<0.01*dy           
        # direct forcing u & v values for xB_E, xB_W, yB_N, yB_S
        pe = jnp.where(xB_e, pEbc, pe)
        uw, vw = jnp.where(xB_w, uWbc, uw), jnp.where(xB_w, vWbc, vw)
        vn = jnp.where(yB_n, vNbc, vn)
        vs = jnp.where(yB_s, vSbc, vs)  
       

        source_x = (pe - pw)
        source_y = (pn - ps)

      
        res_u = ( source_x - (uE + uW + uN + uS)/(dx*Re))/dx 
        res_v = ( source_y - (vE + vW + vN + vS)/(dx*Re))/dx

        res_c = (ue - uw + vn - vs)/dx
        res_p = pE + pW + pN + pS
                
        outputs = jnp.hstack([u, v, p,res_u,res_v,res_c,res_p]) 
        return outputs    
    
class DNN_ad(nn.Module):
    """interploration DNNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]     


    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[-BS_ad:,0:1], inputs[-BS_ad:,1:2]
        
        def get_uvp(x, y):
            # inputs = jnp.hstack([2*(x-x_l)/(x_u-x_l)-1,2*(y-y_l)/(y_u-y_l)-1])
            inputs = jnp.hstack([x,y ])

            # feature mapping
            hidden = self.feats(inputs)
            hidden = jnp.sin(jnp.pi*hidden)
            # hidden = jnp.sin(hidden)

            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p) 


            return (u, v, p) 
        
        u,v,p = get_uvp(x, y)

        def get_uvp_xy(get_uvp, x, y):
            u_x, v_x, p_x = jacfwd(get_uvp)(x, y)
            u_y, v_y, p_y = jacfwd(get_uvp, argnums=1)(x, y)

            return u_x, u_y, v_x, v_y, p_x , p_y
            
        get_uvp_xy_vmap = vmap(get_uvp_xy, in_axes=(None, 0, 0))
        u_x, u_y, v_x, v_y, p_x , p_y = get_uvp_xy_vmap(get_uvp, x,y)
        u_x, u_y, v_x, v_y, p_x , p_y = u_x[:,:,0], u_y[:,:,0], v_x[:,:,0], v_y[:,:,0], p_x[:,:,0] , p_y[:,:,0]

       # obtain u_xx, u_yy, v_xx, v_yy, p_xx, p_yy
        def get_uvp_xxyy(get_uvp, x, y):
            u_xx, v_xx, _= jacfwd(jacfwd(get_uvp))(x, y)
            u_yy, v_yy, _= jacfwd(jacfwd(get_uvp, argnums=1), argnums=1)(x, y)

            return u_xx, u_yy, v_xx, v_yy
        
        get_uvp_xxyy_vmap = vmap(get_uvp_xxyy, in_axes=(None, 0, 0))
        u_xx, u_yy, v_xx, v_yy = get_uvp_xxyy_vmap(get_uvp, x, y)
        u_xx, u_yy, v_xx, v_yy = u_xx[:,:,0,0], u_yy[:,:,0,0], v_xx[:,:,0,0], v_yy[:,:,0,0]
        
        residuals_continuity = u_x + v_y
        residuals_momentum_1 = u*u_x + v*u_y + p_x - 1.0/Re*(u_xx + u_yy)
        residuals_momentum_2 = u*v_x + v*v_y + p_y - 1.0/Re*(v_xx + v_yy)
            
        outputs = jnp.hstack([u, v, p,residuals_continuity,residuals_momentum_1,residuals_momentum_2])

        return outputs     

In [23]:
# choose seed
seed = 10
key, rng = random.split(random.PRNGKey(seed))

# dummy input
a = random.normal(key, [1,2])

# initialization call
bi_count = 1
n_nodes = 32
model, model_0,model_ad = PINN(), DNN(),DNN_ad()
params = model.init(key, a) 
num_params, format_params_fn = get_params_format_fn(params)
print (num_params)

# flatten initial params
params = jax.flatten_util.ravel_pytree(params)[0]  

params_0 = params 


2025-09-08 14:56:10.648619: E external/xla/xla/service/hlo_lexer.cc:438] Failed to parse int literal: 894515288310727292233


12928


In [24]:
point_ad.shape

(204, 2)

In [25]:
point_ad_value = jnp.zeros((len(point_ad[:,0]),3))
print('data_X shape:',data_X.shape,'data_Y shape:',data_Y.shape)
print('data_X_l_BC shape:',data_X_l_BC.shape,'data_Y_l_BC shape:',data_Y_l_BC.shape)
print('data_X_r_BC shape:',data_X_r_BC.shape,'data_Y_r_BC shape:',data_Y_r_BC.shape)
print('data_X_tb_BC shape:',data_X_tb_BC.shape,'data_Y_tb_BC shape:',data_Y_tb_BC.shape)
print('data_X_airfoil_BC shape:',data_X_airfoil_BC.shape,'data_Y_airfoil_BC shape:',data_Y_airfoil_BC.shape)
print('point_ad shape:',point_ad.shape,'point_ad_value shape:',point_ad_value.shape)

data_X shape: (320190, 2) data_Y shape: (320190, 3)
data_X_l_BC shape: (401, 2) data_Y_l_BC shape: (401, 3)
data_X_r_BC shape: (401, 2) data_Y_r_BC shape: (401, 3)
data_X_tb_BC shape: (1602, 2) data_Y_tb_BC shape: (1602, 3)
data_X_airfoil_BC shape: (400, 2) data_Y_airfoil_BC shape: (400, 3)
point_ad shape: (204, 2) point_ad_value shape: (204, 3)


In [26]:
@jit
def minibatch(key):
    key1, key2, key3, key4, key5,key6 = key
    batch_all = random.choice(key1, n_all , (BS_PDE,),replace = False)
    batch_bc_l = random.choice(key2, n_l_BC, (BS_l_BC,),replace = False)   
    batch_bc_r = random.choice(key3, n_r_BC, (BS_r_BC,),replace = False)   
    batch_bc_tb = random.choice(key4, n_tb_BC, (BS_tb_BC,),replace = False)   
    batch_bc_airfoil = random.choice(key5, n_airfoil_BC, (BS_airfoil_BC,),replace = False)
    batch_ad = random.choice(key6, n_ad, (BS_ad,),replace = False)
    batch_X = jnp.vstack([data_X[batch_all], 
                          data_X_l_BC[batch_bc_l], 
                          data_X_r_BC[batch_bc_r],
                          data_X_tb_BC[batch_bc_tb],
                          data_X_airfoil_BC[batch_bc_airfoil],
                          point_ad[batch_ad],
                          ])
    batch_Y = jnp.vstack([data_Y[batch_all], 
                          data_Y_l_BC[batch_bc_l],
                          data_Y_r_BC[batch_bc_r], 
                          data_Y_tb_BC[batch_bc_tb],
                          data_Y_airfoil_BC[batch_bc_airfoil],
                          point_ad_value[batch_ad],
                          ])
    
    return (batch_X, batch_Y)


In [27]:
# loss function
def eval_loss(params,params_0, inputs, labels):
    pred = model.apply(format_params_fn(params), inputs)
    u_pinn, v_pinn, p_pinn, residuals_continuity, residuals_momentum_1, residuals_momentum_2, uv_bc,p_bc, nbc,res_u,res_v,top_bc_u,bottom_bc_u,l_bc,t_bc,b_bc,res_c,res_p = jnp.split(pred, 18, axis=1)
    gt_u, gt_v,_ = jnp.split(labels, 3, axis=1)
    pred0 = model_0.apply(format_params_fn(params_0), inputs)
    u_0, v_0, p_0,res_u0,res_v0,res_c0,res_p0 = jnp.split(pred0, 7, axis=1)
    pred_ad = model_ad.apply(format_params_fn(params), inputs)
    u_ad,v_ad,p_ad,ad_c,ad_m1,ad_m2 = jnp.split(pred_ad, 6, axis=1) 
    
    gt_u = gt_u[:-BS_ad]
    gt_v = gt_v[:-BS_ad]
    # gt_p = p[:-BS_ad]
    u = u_pinn[:-BS_ad]
    v = v_pinn[:-BS_ad]
    p = p_pinn[:-BS_ad]
    residuals_continuity = residuals_continuity[:-BS_ad]
    residuals_momentum_1 = residuals_momentum_1[:-BS_ad]
    residuals_momentum_2 = residuals_momentum_2[:-BS_ad]
    uv_bc = uv_bc[:-BS_ad]
    p_bc = p_bc[:-BS_ad]
    nbc = nbc[:-BS_ad]
    res_u = res_u[:-BS_ad]
    res_v = res_v[:-BS_ad]
    res_c = res_c[:-BS_ad]
    res_p = res_p[:-BS_ad]
    top_bc_u = top_bc_u[:-BS_ad]
    bottom_bc_u = bottom_bc_u[:-BS_ad]
    l_bc = l_bc[:-BS_ad]
    t_bc = t_bc[:-BS_ad]
    b_bc = b_bc[:-BS_ad]
    u_0 = u_0[:-BS_ad]
    v_0 = v_0[:-BS_ad]
    p_0 = p_0[:-BS_ad]
    res_u0 = res_u0[:-BS_ad]
    res_v0 = res_v0[:-BS_ad]
    res_c0 = res_c0[:-BS_ad]
    res_p0 = res_p0[:-BS_ad]
    

    beta = 0.95

    p_coe = 4/Re
    loss_u = (u - (beta*(-residuals_momentum_1 + res_u0 - res_u)*dx*dx*Re/4) - u_0)
    loss_v = (v - (beta*(-residuals_momentum_2 + res_v0 - res_v)*dx*dx*Re/4) - v_0)
    loss_p = (p - beta*(res_p - res_p0 -p_coe*res_c )/4 - p_0)


    pde_uvp  = jnp.square(res_c) *15 + jnp.square(residuals_momentum_1) + jnp.square(residuals_momentum_2)
    uv_rc =  jnp.abs(loss_u) + jnp.abs(loss_v)  + jnp.abs(loss_p) 

    pde_uvp = pde_uvp[:-BS_airfoil_BC]
    uv_rc = uv_rc[:-BS_airfoil_BC]
    nbc = nbc[:-BS_airfoil_BC]
    
    
    pde_uvp_ad  = jnp.mean(jnp.square(ad_c)) *1 + jnp.mean(jnp.square(ad_m1)) + jnp.mean(jnp.square(ad_m2))

    
    pde_loss = jnp.sum(pde_uvp*nbc) / nbc.sum()
    rc_loss = jnp.sum(uv_rc*nbc) / nbc.sum()
    
            
    bc_u = (u - 1.0)
    bc_v = (v)
    bc_uv_loss = jnp.sum(jnp.square(bc_v)*uv_bc) / uv_bc.sum()  + jnp.sum(jnp.square(bc_u)*l_bc) / l_bc.sum() + jnp.sum(jnp.square(top_bc_u)*t_bc) / t_bc.sum()+ jnp.sum(jnp.square(bottom_bc_u)*b_bc) / b_bc.sum()
    
    
    bc_p = p  # p_bc is the mask for p boundary condition
    bc_p_loss =  jnp.sum(jnp.square(bc_p)*p_bc) / p_bc.sum()  # p_bc is the mask for p boundary condition
    
    bc_u_airfoil = u[-BS_airfoil_BC:]
    bc_v_airfoil = v[-BS_airfoil_BC:]  # airfoil_BC is the mask for airfoil boundary condition
    bc_airfoil = jnp.mean(jnp.square(bc_u_airfoil)) + jnp.mean(jnp.square(bc_v_airfoil))  # airfoil_BC is the mask for airfoil boundary condition
    bc_loss = bc_uv_loss + bc_p_loss + bc_airfoil*1  # airfoil_BC is the mask for airfoil boundary condition
    
    loss = pde_loss*1 + bc_loss*1 + rc_loss*20 + pde_uvp_ad*0.01
    
    gt_V = jnp.sqrt(gt_u**2 + gt_v**2)  # ground truth velocity magnitude
    V = jnp.sqrt(u**2 + v**2)  # predicted velocity magnitude
    mse_u = jnp.mean(jnp.square(u - gt_u)) 
    mse_v = jnp.mean(jnp.square(v - gt_v)) 
    mse_V = jnp.mean(jnp.square(V - gt_V))  # mean squared error of velocity magnitude
    l2_u = jnp.linalg.norm(u - gt_u) / jnp.linalg.norm(gt_u)
    l2_v = jnp.linalg.norm(v - gt_v) / jnp.linalg.norm(gt_v)
    l2_V = jnp.linalg.norm(V - gt_V) / jnp.linalg.norm(gt_V)  # relative l2 error of velocity magnitude
    
    
    return loss, (mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,pde_uvp_ad)

loss_grad = jax.jit(jax.value_and_grad(eval_loss, has_aux=True))    

In [28]:
# weights update  
@jit
def update(params, params_0,opt_state, key):
    batch_X, batch_Y = minibatch(key)
    (loss, (mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss)), grad = loss_grad(params,params_0, batch_X, batch_Y)
    updates, opt_state = optimizer.update(grad, opt_state)
    params_0 = params # update u_0

    params = optax.apply_updates(params, updates)
    return params, params_0,opt_state, loss,mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss

In [29]:
# optimizer
max_iters = 50000
max_lr = 3e-3
lr_scheduler = optax.warmup_cosine_decay_schedule(init_value=max_lr, peak_value=max_lr, warmup_steps=int(0.0*max_iters),  
                                                  decay_steps=max_iters, end_value=1e-10,exponent =1.0)
optimizer = optax.adam(learning_rate=lr_scheduler) # Choose the method
opt_state = optimizer.init(params)

Training

In [30]:
runtime = 0
train_iters = 0

store = []
while (train_iters <= max_iters):
    # mini-batch update
    start = time.time()
    key1, key2,key3,key4,key5,key6, rng = random.split(rng, 7) # update random generator
    params,params_0, opt_state, loss, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss = update(params,params_0, opt_state, (key1, key2,key3,key4,key5,key6))
    end = time.time()
    runtime += (end-start)    
    # append weights
    if (train_iters % 5000 == 0):
        print ('iter. = %05d,  time = %03ds,  loss = %.2e  |  mse_u = %.2e,  mse_v = %.2e,  mse_V = %.2e,  rl2_u = %.2e,  rl2_v = %.2e,  rl2_V = %.2e, pde =  %.2e, bc = %.2e , rc =  %.2e, interp = %.2e '%(train_iters, runtime, loss, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss))
        store.append([train_iters, runtime, loss, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss])
    train_iters += 1

store = jnp.array(store)

iter. = 00000,  time = 016s,  loss = 9.82e+01  |  mse_u = 2.29e+00,  mse_v = 1.07e-01,  mse_V = 4.16e-01,  rl2_u = 1.51e+00,  rl2_v = 1.76e+01,  rl2_V = 6.41e-01, pde =  9.53e+01, bc = 1.97e+00 , rc =  4.65e-02, interp = 6.26e+00 
iter. = 05000,  time = 056s,  loss = 1.08e-02  |  mse_u = 1.91e-03,  mse_v = 8.45e-05,  mse_V = 1.92e-03,  rl2_u = 4.35e-02,  rl2_v = 5.11e-01,  rl2_V = 4.36e-02, pde =  2.63e-03, bc = 7.36e-04 , rc =  3.21e-04, interp = 1.02e-01 
iter. = 10000,  time = 096s,  loss = 8.55e-03  |  mse_u = 1.94e-03,  mse_v = 8.97e-05,  mse_V = 1.95e-03,  rl2_u = 4.37e-02,  rl2_v = 5.04e-01,  rl2_V = 4.38e-02, pde =  2.30e-03, bc = 8.90e-04 , rc =  2.41e-04, interp = 5.31e-02 
iter. = 15000,  time = 136s,  loss = 6.23e-03  |  mse_u = 7.33e-04,  mse_v = 3.60e-05,  mse_V = 7.40e-04,  rl2_u = 2.69e-02,  rl2_v = 3.40e-01,  rl2_V = 2.71e-02, pde =  1.11e-03, bc = 8.93e-04 , rc =  1.80e-04, interp = 6.28e-02 
iter. = 20000,  time = 177s,  loss = 5.62e-03  |  mse_u = 3.21e-04,  mse_v =

In [31]:
sim = pd.read_csv('naca0012_RE500_801x401.csv',sep=',')
sim_x = sim['x'].values.reshape(-1,1)
sim_y = sim['y'].values.reshape(-1,1)
sim_u = sim['u'].values.reshape(-1,1)
sim_v = sim['v'].values.reshape(-1,1)
sim_p = sim['p'].values.reshape(-1,1)
data_X, data_Y = np.hstack([sim_x, sim_y]), np.hstack([sim_u, sim_v, sim_p])

print('the shape of data before subsampling:')
print('data_X.shape:',data_X.shape)
print('data_Y.shape:',data_Y.shape)
bool_small = (data_X[:,0] <= 1.05) & (data_X[:,0] >= -0.05) & (data_X[:,1] <= 0.2) & (data_X[:,1] >= -0.2)
data_X = data_X[bool_small]
data_Y = data_Y[bool_small]
print('the shape of data after subsampling:')
print('data_X.shape:',data_X.shape)
print('data_Y.shape:',data_Y.shape)
air_bool = filter_points_inside_airfoil(data_X)
data_XX = data_X[~air_bool]
data_YY = data_Y[~air_bool]

# plt.xlim(-0.1, 1.1)
# plt.ylim(-0.1, 0.1)
print('the shape of data_XX after removing the airfoil:')
print('data_XX.shape:',data_XX.shape)
print('data_YY.shape:',data_YY.shape)

x_upper, y_upper = naca0012_x, +y_t
x_lower, y_lower = naca0012_x, -y_t

data_airfoli_bc_xy1 = jnp.hstack([x_upper, y_upper])
data_airfoli_bc_xy2 = jnp.hstack([x_lower, y_lower])
data_airfoli_bc_xy = jnp.vstack([data_airfoli_bc_xy1, data_airfoli_bc_xy2])



the shape of data before subsampling:
data_X.shape: (321201, 2)
data_Y.shape: (321201, 3)
the shape of data after subsampling:
data_X.shape: (4551, 2)
data_Y.shape: (4551, 3)
the shape of data_XX after removing the airfoil:
data_XX.shape: (3744, 2)
data_YY.shape: (3744, 3)


In [32]:
data_X, data_Y = data_XX, data_YY
inputs, labels = data_X, data_Y

print(inputs.shape)
uvp = model.apply(format_params_fn(params), inputs)
u, v, p = uvp[:,0:1], uvp[:,1:2], uvp[:,2:3]
gt_u, gt_v,gt_p = jnp.split(labels, 3, axis=-1)

gt_V = jnp.sqrt(gt_u**2 + gt_v**2)  # ground truth velocity magnitude
V = jnp.sqrt(u**2 + v**2)  # predicted velocity magnitude
mse_u = jnp.mean(jnp.square(u - gt_u)) 
mse_v = jnp.mean(jnp.square(v - gt_v)) 
mse_V = jnp.mean(jnp.square(V - gt_V)) 
# mean squared error of velocity magnitude
l2_u = jnp.linalg.norm(u - gt_u) / jnp.linalg.norm(gt_u)
l2_v = jnp.linalg.norm(v - gt_v) / jnp.linalg.norm(gt_v)
l2_V = jnp.linalg.norm(V - gt_V) / jnp.linalg.norm(gt_V)  # relative l2 error of velocity magnitude

mse_p = jnp.mean(jnp.square(p - gt_p))  # mean squared error of pressure
rl2 = jnp.linalg.norm(p - gt_p) / jnp.linalg.norm(gt_p)  # relative l2 error of pressure

print ('[Re=%.1f] :  mse_u = %.2e,  mse_v = %.2e,  mse_V = %.2e,  rl2_u = %.2e,  rl2_v = %.2e,  rl2_V = %.2e, mse_p = %.2e, rl2_p = %.2e'%(Re, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, mse_p, rl2))


(3744, 2)
[Re=500.0] :  mse_u = 3.54e-05,  mse_v = 7.06e-06,  mse_V = 3.57e-05,  rl2_u = 7.37e-03,  rl2_v = 2.51e-02,  rl2_V = 7.33e-03, mse_p = 2.86e-05, rl2_p = 4.61e-02
